In [ ]:
%pip install -r requirements.txt

In [ ]:
import geopandas as gpd
import os
from pathlib import Path
from dotenv import load_dotenv
import os
import shutil
import random
import math

def zip_with_shutil(folder_path, output_path):
    shutil.make_archive(output_path.replace('.zip', ''), 'zip', folder_path)

# Load environment variables from the .env file
load_dotenv()

# Source directory
source_dir = os.getenv("SOURCE_PATH")
target_dir = os.getenv("TARGET_PATH")
zip_dir = os.getenv("ZIP_PATH")

requires_id = {
    "Creeks": "Creek", 
    "UG_Lakes": "Lake", 
    'UG_MajRoads': 'Road',
    'ComboPlan_ProposedGravelAugmentation': 'CP_PGA',
    'ComboPlan_ProposedRiprap': 'CP_PRR',
    'ComboPlan_SlopeRepair': 'CP_SR',    
}

filter_ids = {
    "Land_Parcels": "APN",
    'ComboPlan_Bridges': 'ID',
    'ComboPlan_ChannelWideningFtprnt': 'ID',
    'ComboPlan_Floodwalls': 'ID',    
    'UG_StrucsOnly_Ind_0824': 'Struc_Name',        
}

generate_cost = {
    "Land_Parcels": "LOTACRES"
}

entries = os.listdir(source_dir)

for entry in entries:
    if entry.endswith('.shp') or entry.endswith('.gdb'):
    
        basename = str(Path(entry).with_suffix(''))
        output_directory = target_dir + "/" + basename
        
        output_path = output_directory + "/"  + basename + ".shp"
        zip_path = zip_dir + "/"  + basename + ".zip"
    
        if not os.path.exists(output_path):
            # Read a specific layer from the GDB
            gdf = gpd.read_file(source_dir + "/" + entry)
            # Convert to 4326
            gdf = gdf.to_crs(epsg=4326)
            # Fix geometries            
            gdf["geometry"] = gdf.geometry.make_valid()
            
            # Add an ID if its needed
            if basename in requires_id:
                ids = []
                
                for idx, row in gdf.iterrows():
                    ids.append(requires_id.get(basename) + '_' + str(idx))
                
                gdf['ID'] = ids
                
            # Add $ value column
            if basename in generate_cost:
                values = []
                
                for idx, row in gdf.iterrows():
                    values.append( math.ceil(random.randint(3000, 7000) * row[generate_cost.get(basename)]))
                
                gdf['Value'] = values
                
            # Filter out rows missing id column if needed
            if basename in filter_ids:
                gdf = gdf[gdf[filter_ids.get(basename)].notna()]            
    
            # Export to Shapefile
    
            os.makedirs(output_directory, exist_ok=True)
    
            gdf.to_file(output_path, driver="ESRI Shapefile")
            
            zip_with_shutil(output_directory, zip_path)
